# CubeAnalysis : toutes les figures

Ce notebook appelle une par une les étapes de figures de `CubeAnalysis` (celles que `run_analysis` enchaîne) et affiche chaque PNG produit directement sous la cellule. Les figures sont donc identiques à celles du pipeline en ligne de commande.

Deux modes :

* **`DEMO = true`** : génère un petit jeu de cubes synthétiques multiphases (champs gaussiens en loi de puissance, densité lognormale, $P/k$ quasi constant) au format attendu, pour que tout tourne sans données.
* **`DEMO = false`** : utilise `INPUT_DIR` (un répertoire contenant `density.fits`, `temperature.fits`, `Bx.fits` ... `Vz.fits`, ou ce que déclare ton fichier de paramètres).

La dernière partie (figures d'ensemble $\xi(\sigma_v)$ et comparaisons) demande plusieurs simulations.

Lance Julia avec plusieurs threads (`JULIA_NUM_THREADS=auto`) avant d'ouvrir le kernel.

## 0. Environnement

In [ ]:
using Pkg

# Chemin du dépôt CubeAnalysis. Laisse vide pour une détection automatique,
# sinon mets le chemin, ex. REPO_PATH = "/Users/<toi>/CubeAnalysis"
REPO_PATH = ""

is_cubeanalysis(dir) = (p = joinpath(dir, "Project.toml"); isfile(p) &&
    get(Pkg.TOML.parsefile(p), "name", "") == "CubeAnalysis")

function find_cubeanalysis()
    candidates = String[]
    isempty(REPO_PATH) || push!(candidates, expanduser(REPO_PATH))
    haskey(ENV, "CUBEANALYSIS_DIR") && push!(candidates, expanduser(ENV["CUBEANALYSIS_DIR"]))
    dir = abspath(pwd())                       # le dossier courant et ses parents
    while true
        push!(candidates, dir, joinpath(dir, "CubeAnalysis"), joinpath(dir, "cubeanalysis"))
        parent = dirname(dir); parent == dir && break; dir = parent
    end
    for base in (homedir(), joinpath(homedir(), "Desktop"), joinpath(homedir(), "Documents"),
                 joinpath(homedir(), ".julia", "dev"))
        push!(candidates, joinpath(base, "CubeAnalysis"), joinpath(base, "cubeanalysis"))
    end
    for c in candidates
        isdir(c) && is_cubeanalysis(c) && return normpath(c)
    end
    error("Dépôt CubeAnalysis introuvable depuis $(pwd()). Renseigne REPO_PATH ci-dessus " *
          "(dossier qui contient le Project.toml de CubeAnalysis).")
end

REPO = find_cubeanalysis()
@info "Dépôt CubeAnalysis" REPO
Pkg.activate(REPO)
Pkg.instantiate()

using CubeAnalysis
using CairoMakie
using FITSIO
using FFTW
using Random
using Statistics
const CA = CubeAnalysis
@info "CubeAnalysis chargé" version=pkgversion(CubeAnalysis) threads=Threads.nthreads()

## 1. Paramètres

In [ ]:
DEMO        = true                                   # false pour tes propres cubes
CONFIG_PATH = joinpath(REPO, "config", "cube_analysis.toml")
INPUT_DIR   = "/chemin/vers/ta/simulation"           # utilisé si DEMO = false
OUTPUT_DIR  = joinpath(pwd(), "cubeanalysis_figures")
FORMATS     = ["png"]                                # ajoute "pdf" ou "svg" si besoin ; png requis pour l'affichage
STRIDE      = nothing                                # nothing = valeur du fichier ; 1 pour tout garder

# Mode démo
DEMO_N      = 64                                     # taille des cubes synthétiques
DEMO_DIR    = joinpath(pwd(), "cubeanalysis_demo_data")

# Figures d'ensemble (section 21) : liste de répertoires de simulations
ENSEMBLE_DIRS = String[]                             # rempli automatiquement en mode démo

## 2. Cubes synthétiques (mode démo)

Chaque simulation démo contient les huit cubes attendus par la configuration par défaut :

* $\mathbf{v}$ : champ gaussien de spectre $E(k)\propto k^{-5/3}$, dispersion $\sigma_v$ (km/s) ;
* $\mathbf{B} = B_0\,\hat z + \delta\mathbf{B}$ en mG (unité de la config), $B_0 = 5\,\mu$G ;
* $n = n_0\exp(\sigma_s s - \sigma_s^2/2)$ avec $s$ gaussien, donc PDF lognormale ;
* $T = (P/k)/n$ avec $P/k \approx 3000$ K cm$^{-3}$ et une dispersion lognormale, ce qui peuple CNM, UNM et WNM.

Trois simulations avec $\sigma_v = 2, 5, 10$ km/s alimentent les figures d'ensemble.

In [ ]:
using FFTW, FITSIO, Random, Statistics

function gaussian_field(N, slope3d; rng)
    k = fftfreq(N, N)
    F = fft(randn(rng, N, N, N))
    @inbounds for I in CartesianIndices(F)
        kk = sqrt(k[I[1]]^2 + k[I[2]]^2 + k[I[3]]^2)
        F[I] *= kk == 0 ? 0.0 : kk^(slope3d / 2)
    end
    g = real(ifft(F))
    return (g .- mean(g)) ./ std(g)
end

function write_fits_cube(path, cube)
    FITS(path, "w") do f
        write(f, Float32.(cube))
    end
end

function make_demo_simulation(dir; N=64, sigma_v=5.0, seed=1)
    mkpath(dir)
    rng = MersenneTwister(seed)
    kolmo = -11 / 3                      # spectre 3D -> E(k) ~ k^-5/3
    vx, vy, vz = (sigma_v .* gaussian_field(N, kolmo; rng) for _ in 1:3)
    B0, dB = 5.0e-3, 2.0e-3              # mG
    bx = dB .* gaussian_field(N, kolmo; rng)
    by = dB .* gaussian_field(N, kolmo; rng)
    bz = B0 .+ dB .* gaussian_field(N, kolmo; rng)
    sigma_s = 0.6 + 0.12 * sigma_v       # contraste plus fort quand sigma_v croît
    s = gaussian_field(N, -3.2; rng)
    density = 1.0 .* exp.(sigma_s .* s .- sigma_s^2 / 2)
    p_over_k = 3000.0 .* exp.(0.3 .* gaussian_field(N, -3.2; rng))
    temperature = p_over_k ./ density
    for (name, cube) in ("density" => density, "temperature" => temperature,
                         "Bx" => bx, "By" => by, "Bz" => bz,
                         "Vx" => vx, "Vy" => vy, "Vz" => vz)
        write_fits_cube(joinpath(dir, "$name.fits"), cube)
    end
    return dir
end

if DEMO
    demo_sims = [make_demo_simulation(joinpath(DEMO_DIR, "sim_sigma" * lpad(round(Int, sv), 2, '0'));
                                      N=DEMO_N, sigma_v=sv, seed=i)
                 for (i, sv) in enumerate((2.0, 5.0, 10.0))]
    INPUT_DIR = demo_sims[2]
    isempty(ENSEMBLE_DIRS) && append!(ENSEMBLE_DIRS, demo_sims)
    @info "Cubes démo écrits" DEMO_DIR demo_sims
end

## 3. Configuration et chargement des champs

In [ ]:
cfg = CA.load_config(CONFIG_PATH)
cfg["run"]["input_dir"]  = abspath(INPUT_DIR)
cfg["run"]["output_dir"] = abspath(OUTPUT_DIR)
cfg["run"]["formats"]    = copy(FORMATS)
"png" in FORMATS || push!(cfg["run"]["formats"], "png")
STRIDE === nothing || (cfg["run"]["sample_stride"] = STRIDE)
DEMO && (cfg["run"]["sample_stride"] = 1)        # cubes 64^3 : pas besoin de sous-échantillonner
mkpath(cfg["run"]["output_dir"])

OUT     = cfg["run"]["output_dir"]
FMT     = string.(cfg["run"]["formats"])
OVER    = true
STRIDE_ = Int(get(cfg["run"], "sample_stride", 1))

fields, metadata = CA.load_fields(cfg)
@info "Champs prêts" size=CA.analysis_field_size(fields, first(keys(fields))) fields=sort!(collect(keys(fields))) lazy=(fields isa CA.LazyFieldStore)

Petit utilitaire : `stage(nom) do files ... end` exécute une étape, mesure son temps, affiche les PNG produits et, comme le pipeline, signale une étape impossible (champ absent, etc.) sans arrêter le notebook.

In [ ]:
struct PNGFile
    path::String
end
Base.show(io::IO, ::MIME"image/png", p::PNGFile) = write(io, read(p.path))

const ALL_FILES = String[]

function stage(action, name::AbstractString)
    files = String[]
    t = try
        @elapsed action(files)
    catch err
        @warn "Étape « $name » ignorée" exception=(err, catch_backtrace())
        return files
    end
    append!(ALL_FILES, files)
    pngs = filter(f -> endswith(f, ".png"), files)
    @info "$name : $(length(pngs)) figure(s) en $(round(t; digits=1)) s"
    for p in pngs
        println(basename(p))
        display(PNGFile(p))
    end
    return files
end

## 4. Coupes (`slices`)

Coupes 2D de chaque champ aux positions `[slices].positions` le long de `[slices].axes`. Fichiers `slices_<champ>`.

In [ ]:
stage("slices") do files
    CA.plot_slices!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 5. Projections (`projections`)

Projections moyenne, somme et maximum le long de chaque axe, plus la densité de colonne $N$ pour `column_density_fields`. Fichiers `projections_<champ>`.

In [ ]:
stage("projections") do files
    CA.plot_projections!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 6. Histogrammes (`histograms`)

Un histogramme 1D par champ, plus les groupes `[[histograms.groups]]` (composantes de $\mathbf{B}$ et de $\mathbf{v}$ superposées).

In [ ]:
stage("histograms") do files
    CA.plot_histograms!(files, fields, metadata, cfg, OUT, FMT, OVER, STRIDE_)
end;

## 7. Diagrammes de phase (`phase_diagrams`)

Histogrammes 2D ($n$, $T$) et ($n$, $P/k$) avec isothermes et courbe d'équilibre thermique (`thermal_overlay`).

In [ ]:
stage("phase_diagrams") do files
    CA.plot_phase_diagrams!(files, fields, metadata, cfg, OUT, FMT, OVER, STRIDE_)
end;

## 8. Relations conditionnelles (`relations`)

Médiane et quantiles de $|B|$ et $|v|$ par intervalle de densité.

In [ ]:
stage("relations") do files
    CA.plot_relations!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 9. Spectres de puissance scalaires (`power_spectra`)

Spectres isotropes $E(k)$ des champs de `[spectra].fields`, avec ajustement en loi de puissance dans le domaine inertiel, zones d'injection et de dissipation ombrées, références Kolmogorov et Burgers pour la vitesse.

In [ ]:
stage("power_spectra") do files
    CA.plot_spectra!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 10. Spectres vectoriels (`vector_spectra`)

Spectres de $\mathbf{B}$, $\mathbf{v}$, de la vorticité ($\nabla\times\mathbf{v}$) et de $\sqrt{\rho}\,\mathbf{v}$, avec décomposition solénoïdale / compressive.

In [ ]:
stage("vector_spectra") do files
    CA.plot_vector_spectra!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 11. Spectres croisés (`cross_spectra`)

Spectre croisé et cohérence entre $n$ et $|B|$.

In [ ]:
stage("cross_spectra") do files
    CA.plot_cross_spectra!(files, fields, cfg, OUT, FMT, OVER)
end;

## 12. Alignements et $\xi$ (`alignments`)

Distribution de l'angle entre $\nabla n$ et $\mathbf{B}$ par intervalle de densité (HRO, paramètre $\zeta$), puis la figure `xi_<nom>` : $\xi_{\nabla n,B}$ et $\xi_{\delta v,B}$ en fonction de $n$ (`xi_plot = true`).

In [ ]:
stage("alignments") do files
    CA.plot_alignments!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 13. Table physique (`physics_table`)

Table-figure : $\mathcal{M}_s$, $\mathcal{M}_A$, $\beta$, $\delta B/|\langle B\rangle|$, $\delta v/|\langle v\rangle|$ (moyennes pondérées volume et masse).

In [ ]:
stage("physics_table") do files
    CA.plot_physics_table!(files, fields, cfg, OUT, FMT, OVER)
end;

## 14. PDF de densité (`density_pdf`)

PDF de $s=\ln(n/\langle n\rangle)$, ajustement lognormal et paramètre de forçage $b$ (avec et sans correction magnétique).

In [ ]:
stage("density_pdf") do files
    CA.plot_density_pdf!(files, fields, cfg, OUT, FMT, OVER)
end;

## 15. Bilan des phases (`phase_budget`)

Fractions de volume et de masse par phase thermique (CNM, UNM, WNM).

In [ ]:
stage("phase_budget") do files
    CA.plot_phase_budget!(files, fields, cfg, OUT, FMT, OVER)
end;

## 16. PDF de densité de colonne (`column_density_pdf`)

N-PDF selon chaque axe de `column_density_axes`, avec ajustement de la queue en loi de puissance.

In [ ]:
stage("column_density_pdf") do files
    CA.plot_column_density_pdf!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 17. Fonctions de structure et ESS par phase (`advanced_diagnostics`)

Fonctions de structure d'ordres `structure_orders` pour `structure_fields` (`structure_<champ>`), puis exposants ESS $\zeta_p/\zeta_3$ de la vitesse par phase thermique comparés à K41, She-Lévêque et Boldyrev (`phase_structure_ess`).

In [ ]:
stage("advanced_diagnostics") do files
    CA.plot_advanced_diagnostics!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 18. Diagnostics par phase (`phase_diagnostics`)

Fractions de phase, grandeurs physiques par phase ($\mathcal{M}_s$, $\mathcal{M}_A$, $\beta$) avec bootstrap par blocs, intermittence et PDF d'incréments de vitesse par phase.

In [ ]:
stage("phase_diagnostics") do files
    CA.plot_phase_diagnostics!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 19. Observations synthétiques (`synthetic_observations`)

Pour chaque ligne de visée : moments (centroïde, dispersion), cube PPV, carte de polarisation synthétique (densité de colonne avec orientation du champ projeté, fraction $p$), puis N-PDF comparées et relation largeur de raie / taille.

In [ ]:
stage("synthetic_observations") do files
    CA.plot_synthetic_observations!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 20a. Topologie (`topology`)

Fonctionnelles de Minkowski et statistiques des structures connexes au-dessus des seuils en quantiles de densité.

In [ ]:
stage("topology") do files
    CA.plot_topology!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 20b. Spectres anisotropes (`anisotropic_spectra`)

Spectre 2D $P(k_\parallel, k_\perp)$ de la densité relativement à $\langle\mathbf{B}\rangle$.

In [ ]:
stage("anisotropic_spectra") do files
    CA.plot_anisotropic_spectra!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 20c. Fonctions de structure directionnelles (`directional_structure_functions`)

Fonctions de structure de $\mathbf{v}$ (et $\rho^{1/3}\mathbf{v}$) longitudinales et transverses, avec exposants ajustés.

In [ ]:
stage("directional_structure_functions") do files
    CA.plot_directional_structure!(files, fields, metadata, cfg, OUT, FMT, OVER)
end;

## 21. Figures d'ensemble (plusieurs simulations)

Ces deux figures proviennent de `scripts/run_simu_ramses.jl` et comparent des simulations entre elles :

* `xi_vs_sigma_v` : $\xi_{\nabla n,B}$ et $\xi_{\delta v,B}$ globaux en fonction de $\sigma_v$ ;
* `ensemble_physical_scaling`, `ensemble_phase_fractions`, `ensemble_phase_zeta2` : $\mathcal{M}_s$, $\mathcal{M}_A$, $\beta$, pente spectrale, longueur de corrélation, fractions de phase et $\zeta_2/\zeta_3$ par phase en fonction de $\sigma_v$.

Chaque répertoire de `ENSEMBLE_DIRS` doit contenir les huit FITS standard (`density`, `temperature`, `Bx`...`Vz`).

In [ ]:
if length(ENSEMBLE_DIRS) >= 2
    ens_out = joinpath(OUT, "ensemble"); mkpath(ens_out)
    grid = get(cfg, "grid", Dict())
    bs = get(grid, "box_size", 1.0)
    box = bs isa AbstractVector ? Tuple(Float64.(bs)) : (Float64(bs), Float64(bs), Float64(bs))
    order = Tuple(string.(get(grid, "axis_order", ["x", "y", "z"])))

    stage("xi_vs_sigma_v") do files
        CA.plot_xi_vs_sigma_v!(files, ENSEMBLE_DIRS, ens_out, FMT; overwrite=true,
                               box_size=box, axis_order=order)
    end
    stage("ensemble_comparison") do files
        CA.plot_ensemble_comparison!(files, ENSEMBLE_DIRS, ens_out, FMT; overwrite=true,
                                     box_size=box, axis_order=order)
    end
else
    @info "Figures d'ensemble ignorées : il faut au moins deux répertoires dans ENSEMBLE_DIRS"
end;

## 22. Récapitulatif

In [ ]:
pngs = sort!(filter(f -> endswith(f, ".png"), unique(ALL_FILES)))
println(length(pngs), " figures PNG dans ", OUT)
foreach(p -> println("  ", relpath(p, OUT)), pngs)

Remarques :

* Les étapes `summary` et `quality` ne produisent que des tables ; elles tournent avec `cfg["run"]["save_data"] = true` via `CA.write_summary!` et `CA.write_quality_report!`. Avec `save_data = true`, les autres étapes écrivent aussi leurs CSV/TOML.
* Pour reproduire exactement une exécution complète en ligne de commande : `CA.run_analysis(CONFIG_PATH)`.